In [1]:
from dataloader import samlet_df
from dataloader import (
    skjorte_data, shorts_data, bukse_data, tshirt_data, 
    langærmet_data, jakke_data, fleece_data, overall_data, 
    forklæde_data, kittel_data, busseron_data, kokkejakke_data,
    andre_data
)

Kategori
Bukser          73373
T-shirt         69384
Jakke           18129
Skjorte         17586
Overall         10605
Langærmet        7380
Busseron         6697
Kokkejakke       5687
Fleece           4429
Andet            3979
Forklæde         3195
Polo             2983
Kittel           2480
tunika           1779
Shorts           1531
cardigan         1144
jeans             989
softshell         718
undertrøje        646
bluse             346
nederdel          343
sjælevarm         252
fusion shirt      227
knickers           91
ziptrøje            7
Name: count, dtype: int64


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import widgets
from IPython.display import display

SURV_SCALE_CONFIG = {
    'Dage':    {'divisor': 1,      'label': 'Dage i cirkulation'},
    'Måneder': {'divisor': 30.437, 'label': 'Måneder i cirkulation'},
    'År':      {'divisor': 365.25, 'label': 'År i cirkulation'},
}
SURV_DATASET_LABELS = {
    'Samlet':     'Samlet_df',      'Skjorte':    'skjorte_data',
    'Shorts':     'shorts_data',    'Bukser':      'bukse_data',
    'T-shirt':    'tshirt_data',    'langærmet':   'langærmet_data',
    'Jakke':      'jakke_data',     'Fleece':       'fleece_data',
    'Forklæde':   'forklæde_data',  'Kittel':       'kittel_data',
    'Busseron':   'busseron_data',  'Kokkejakke':   'kokkejakke_data',
    'Overall':    'overall_data',   'Andet':        'andre_data',
}
SURV_DEFAULT_MAX = int(8 * 365.25)


def interactive_overdødelighed_ssi(Samlet_df=None, skjorte_data=None, shorts_data=None,
                                    bukse_data=None, tshirt_data=None, langærmet_data=None,
                                    jakke_data=None, fleece_data=None, forklæde_data=None,
                                    kittel_data=None, busseron_data=None,
                                    kokkejakke_data=None, overall_data=None, andre_data=None):

    _locs = locals()
    _all  = {v: _locs[v] for v in SURV_DATASET_LABELS.values() if v in _locs}
    available = {lbl: _all[var] for lbl, var in SURV_DATASET_LABELS.items()
                 if _all.get(var) is not None}
    if not available:
        print("Ingen datasæt fundet."); return

    W = 260
    def sep(): return widgets.HTML("<hr style='margin:4px 0'>")
    def lbl(t): return widgets.HTML(f"<small><b>{t}</b></small>")
    def mk_layout(w=W): return widgets.Layout(width=f'{w}px')

    def get_arsager(name):
        return sorted(available[name]['Kassationsårsag (ui)'].dropna().unique())
    def get_kategorier(name):
        if 'Kategori' not in available[name].columns: return []
        return sorted(available[name]['Kategori'].dropna().unique().tolist())
    def dage_max(name):
        return int(available[name]['Dage i cirkulation'].max()) + 100

    # ── Widgets ───────────────────────────────────────────────────────────────
    ds_w = widgets.Dropdown(options=list(available), value=list(available)[0],
        description='Datasæt:', style={'description_width':'70px'}, layout=mk_layout())

    _init_kats = get_kategorier(ds_w.value)
    kat_w = widgets.SelectMultiple(options=_init_kats, value=_init_kats,
        description='', layout=widgets.Layout(width='260px', height='90px'))
    kat_alle_btn = widgets.Button(description='Vælg alle',
        layout=widgets.Layout(width='95px', height='26px'))
    kat_alle_btn.on_click(lambda _: setattr(kat_w, 'value', list(kat_w.options)))
    kat_section = widgets.VBox(
        [lbl('Kategorier (ctrl+klik)'), kat_w, kat_alle_btn],
        layout=widgets.Layout(display='' if _init_kats else 'none'))

    subquery_w = widgets.Text(description='Produkt:',
        placeholder='f.eks. Tshirt, Polo  (komma = OR)',
        value='', style={'description_width':'60px'},
        layout=widgets.Layout(width=f'{W}px'))

    arsag_cbs = {}
    arsag_box = widgets.VBox([])

    skala_w = widgets.ToggleButtons(options=['Dage','Måneder','År'], value='År',
        description='X-skala:', style={'description_width':'70px','button_width':'70px'})

    _mx = dage_max(ds_w.value)
    def mk_float(desc, val, mn, mx, step):
        dec = 0 if step >= 10 else (1 if step >= 1 else 2)
        return widgets.FloatSlider(value=val, min=mn, max=mx, step=step,
            description=desc, readout_format=f'.{dec}f',
            style={'description_width':'55px'}, layout=mk_layout())

    min_dage_w = mk_float('Min:', 0, 0, _mx, 50)
    max_dage_w = mk_float('Max:', min(SURV_DEFAULT_MAX, _mx), 0, _mx, 50)

    bins_w   = widgets.IntSlider(value=60, min=10, max=200, step=5,
        description='Bins:', style={'description_width':'55px'}, layout=mk_layout())
    smooth_w = widgets.IntSlider(value=15, min=2, max=60, step=1,
        description='Glatning:', style={'description_width':'60px'}, layout=mk_layout())
    log_y_w  = widgets.Checkbox(value=False, description='Log y-akse',
        indent=False, layout=mk_layout())

    prev_skala = {'v': 'År'}
    plot_out   = widgets.Output()

    # ── Redraw ────────────────────────────────────────────────────────────────
    def redraw(*_):
        div   = SURV_SCALE_CONFIG[skala_w.value]['divisor']
        min_d = round(min_dage_w.value * div)
        max_d = round(max_dage_w.value * div)

        df = available[ds_w.value].copy()
        if ('Kategori' in df.columns and list(kat_w.options)
                and list(kat_w.value) != list(kat_w.options)):
            df = df[df['Kategori'].isin(kat_w.value)]
        q = subquery_w.value.strip()
        if q:
            terms = [t.strip() for t in q.split(',') if t.strip()]
            mask = df['Produkt - Produkt'].str.contains(terms[0], regex=False, case=False)
            for t in terms[1:]:
                mask = mask | df['Produkt - Produkt'].str.contains(t, regex=False, case=False)
            df = df[mask]
        df = df[(df['Dage i cirkulation'] >= min_d) & (df['Dage i cirkulation'] <= max_d)]
        valgte = [a for a, cb in arsag_cbs.items() if cb.value]
        df_valgt = df[df['Kassationsårsag (ui)'].isin(valgte)] if valgte else df.iloc[:0]

        with plot_out:
            plot_out.clear_output(wait=True)

            if len(df_valgt) == 0:
                fig, ax = plt.subplots(figsize=(10, 5))
                ax.text(0.5, 0.5, 'Ingen data i det valgte interval',
                        ha='center', va='center', transform=ax.transAxes,
                        fontsize=13, color='gray')
                display(fig); plt.close(fig); return

            # ── Beregn histogram: antal kassationer per bin ────────────────
            n_bins = bins_w.value
            bins   = np.linspace(min_d, max_d, n_bins + 1)
            counts, _ = np.histogram(df_valgt['Dage i cirkulation'].dropna().values, bins=bins)
            bin_centers = (bins[:-1] + bins[1:]) / 2
            x = bin_centers / div  # konverter til valgt skala

            # ── Baseline: glidende gennemsnit ──────────────────────────────
            sw = min(smooth_w.value, len(counts))
            kernel   = np.ones(sw) / sw
            baseline = np.convolve(counts.astype(float), kernel, mode='same')

            # ── Glidende std ───────────────────────────────────────────────
            half = sw // 2
            std = np.array([
                np.std(counts[max(0, i-half):min(len(counts), i+half+1)], ddof=0)
                for i in range(len(counts))
            ])
            std = np.where(std < 1e-9, 1e-9, std)

            # ── Plot ───────────────────────────────────────────────────────
            fig, ax = plt.subplots(figsize=(10, 5))

            # Tærskellinjer (stiplet som SSI)
            ax.plot(x, baseline + 4*std, color='#8B0000', lw=1.4,
                    ls='-.', alpha=0.85, label='Tærskel 4 z-score')
            ax.plot(x, baseline + 2*std, color='#c0392b', lw=1.4,
                    ls='--', alpha=0.85, label='Tærskel 2 z-score')

            # Forventet baseline (lyseblå som SSI)
            ax.plot(x, baseline, color='#5aabff', lw=2.2,
                    label='Forventet')

            # Registreret antal (mørkeblå som SSI)
            ax.plot(x, counts, color='#1a237e', lw=1.8,
                    alpha=0.92, label='Registreret')

            # Fremhæv over-tærskel i orange (som SSI's "korr. for forsinkelse")
            over2 = counts > (baseline + 2*std)
            if over2.any():
                ax.fill_between(x, baseline + 2*std, counts,
                                where=over2, alpha=0.45,
                                color='#e67e22', label='Over tærskel')

            titel = ds_w.value + (f' ({q})' if q else '')
            ax.set_title(f'Overdødelighed — {titel}\n{len(df_valgt)} kassationer i alt',
                         fontsize=12)
            ax.set_xlabel(SURV_SCALE_CONFIG[skala_w.value]['label'])
            ax.set_ylabel('Antal kassationer')
            ax.set_xlim(min_dage_w.value, max_dage_w.value)
            ax.legend(fontsize=8, loc='upper right', framealpha=0.9, ncol=2)
            ax.grid(True, alpha=0.2, linestyle='--')

            if log_y_w.value:
                ax.set_yscale('log')

            plt.tight_layout()
            display(fig)
            plt.close(fig)

    # ── Arsag checkboxes ──────────────────────────────────────────────────────
    def build_arsag_cbs(ds_name):
        arsag_cbs.clear()
        rows = []
        for a in get_arsager(ds_name):
            cb = widgets.Checkbox(value=True, description=a, indent=False,
                layout=widgets.Layout(width='260px'))
            arsag_cbs[a] = cb
            cb.observe(redraw, names='value')
            rows.append(cb)
        sel = widgets.Button(description='Vælg alle',
            layout=widgets.Layout(width='95px', height='26px'))
        des = widgets.Button(description='Fravælg alle',
            layout=widgets.Layout(width='105px', height='26px'))
        sel.on_click(lambda _: [setattr(c,'value',True)  for c in arsag_cbs.values()])
        des.on_click(lambda _: [setattr(c,'value',False) for c in arsag_cbs.values()])
        arsag_box.children = rows + [widgets.HBox([sel, des])]

    # ── Slider update ─────────────────────────────────────────────────────────
    def update_sliders(ds_name, skala, preserve=False):
        mx   = dage_max(ds_name)
        div  = SURV_SCALE_CONFIG[skala]['divisor']
        dec  = 0 if skala == 'Dage' else (1 if skala == 'Måneder' else 2)
        step = 50 if skala == 'Dage' else (1 if skala == 'Måneder' else 0.25)
        new_mx = round(mx / div, dec)
        cap    = round(SURV_DEFAULT_MAX / div, dec)
        if preserve:
            old_div  = SURV_SCALE_CONFIG[prev_skala['v']]['divisor']
            new_min  = round(min_dage_w.value * old_div / div, dec)
            new_maxv = round(max_dage_w.value * old_div / div, dec)
        else:
            new_min = 0; new_maxv = min(new_mx, cap)
        for w in [min_dage_w, max_dage_w]:
            w.step = step; w.max = new_mx; w.readout_format = f'.{dec}f'
        min_dage_w.value = max(0, min(new_mx, new_min))
        max_dage_w.value = max(0, min(new_mx, new_maxv))

    def on_skala(change):
        update_sliders(ds_w.value, change['new'], preserve=True)
        prev_skala['v'] = change['new']
    skala_w.observe(on_skala, names='value')

    def on_dataset(change):
        name = change['new']
        update_sliders(name, skala_w.value)
        build_arsag_cbs(name)
        kats = get_kategorier(name)
        kat_w.options = kats; kat_w.value = kats
        kat_section.layout.display = '' if kats else 'none'
    ds_w.observe(on_dataset, names='value')

    for w in [ds_w, skala_w, min_dage_w, max_dage_w, log_y_w, subquery_w, bins_w, smooth_w]:
        w.observe(redraw, names='value')
    kat_w.observe(redraw, names='value')

    build_arsag_cbs(ds_w.value)

    # ── Layout ────────────────────────────────────────────────────────────────
    col_layout = widgets.Layout(width='285px', min_width='285px', padding='6px',
                                border='1px solid #ddd', overflow_y='auto')
    controls = widgets.VBox([
        widgets.HTML("<b style='font-size:13px'>Overdødelighed — Indstillinger</b>"),
        sep(), lbl('Datasæt'), ds_w,
        sep(), kat_section,
        sep(), lbl('Produkt filter'), subquery_w,
        sep(), lbl('Kassationsårsag'), arsag_box,
        sep(), lbl('X-akse skala'), skala_w,
        sep(), lbl('Dage i cirkulation'), min_dage_w, max_dage_w,
        sep(), bins_w, smooth_w, log_y_w,
    ], layout=col_layout)
    plots_panel = widgets.VBox([plot_out],
        layout=widgets.Layout(flex='1', min_width='0', padding='0 0 0 12px'))

    display(widgets.HBox([controls, plots_panel],
        layout=widgets.Layout(width='100%', align_items='flex-start')))
    redraw()


# ── Kald ──────────────────────────────────────────────────────────────────────
interactive_overdødelighed_ssi(
    Samlet_df=samlet_df,
    tshirt_data=tshirt_data,
    skjorte_data=skjorte_data,
    shorts_data=shorts_data,
    bukse_data=bukse_data,
    langærmet_data=langærmet_data,
    jakke_data=jakke_data,
    fleece_data=fleece_data,
    forklæde_data=forklæde_data,
    kittel_data=kittel_data,
    busseron_data=busseron_data,
    kokkejakke_data=kokkejakke_data,
    overall_data=overall_data,
    andre_data=andre_data,
)